## Classification with a Tabular Vector Borne Disease Dataset

**Goal**: The goal of this project is to classify vector-borne diseases using a tabular dataset. You will apply machine learning techniques to predict the disease outcome based on various features such as environmental and demographic data.

**Dataset**: The dataset can be downloaded from [here](https://www.kaggle.com/competitions/playground-series-s3e13/data).


In [20]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
train= pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
train.shape

(707, 66)

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 66 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     707 non-null    int64  
 1   sudden_fever           707 non-null    float64
 2   headache               707 non-null    float64
 3   mouth_bleed            707 non-null    float64
 4   nose_bleed             707 non-null    float64
 5   muscle_pain            707 non-null    float64
 6   joint_pain             707 non-null    float64
 7   vomiting               707 non-null    float64
 8   rash                   707 non-null    float64
 9   diarrhea               707 non-null    float64
 10  hypotension            707 non-null    float64
 11  pleural_effusion       707 non-null    float64
 12  ascites                707 non-null    float64
 13  gastro_bleeding        707 non-null    float64
 14  swelling               707 non-null    float64
 15  nausea

In [6]:
print(len(train.columns))
enc = LabelEncoder()
train['prognosis_enc'] = enc.fit_transform(train['prognosis'])
print(len(train.columns))
train.head()

66
67


,id,sudden_fever,headache,mouth_bleed,nose_bleed,muscle_pain,joint_pain,vomiting,rash,diarrhea,...,toe_inflammation,finger_inflammation,lips_irritation,itchiness,ulcers,toenail_loss,speech_problem,bullseye_rash,prognosis,prognosis_enc
0,0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Lyme_disease,3
1,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tungiasis,7
2,2,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,Lyme_disease,3
3,3,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Zika,10
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,Rift_Valley_fever,6


In [7]:
X = train.drop(['prognosis', 'prognosis_enc'], axis=1)
y = train['prognosis_enc']

In [14]:
X_test = test.drop(['id'], axis=1)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


In [22]:
class Pipeline:
    def __init__(self, model='XGB', **kwargs):
        self.model_type = model
        self.model = self.get_model(model, **kwargs)

    def get_model(self, model, **kwargs):
        models = {
            'Decision Tree': DecisionTreeClassifier,
            'Random Forest': RandomForestClassifier,
            'KNN': KNeighborsClassifier
        }
        if model not in models:
            raise ValueError("Invalid model type")
        return models[model](**kwargs)

    def fit(self, X, y):
        self.model.fit(X, y)
        self.y_train_pred = self.model.predict(X)

    def predict(self, X):
        return self.model.predict(X)

    def accuracy(self, y_true):
        return accuracy_score(y_true, self.y_train_pred)

In [24]:
models = {
    'Decision Tree': {'max_depth': 5},
    'Random Forest': {'n_estimators': 100,'max_depth': 5,'min_samples_leaf': 1,'random_state': 42},
    'KNN': {'n_neighbors': 5}
}

In [26]:
# Train and evaluate models
best_model = None
best_accuracy = 0.0

for model_name, params in models.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline(model=model_name, **params)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = pipeline.accuracy(y_train)
    print(f"Accuracy for {model_name}: {accuracy}")
    print("----------------------------------------")

    # Track the best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model_name

print(f"The best model is {best_model} with accuracy {best_accuracy}")

Training Decision Tree...
Accuracy for Decision Tree: 0.4018867924528302
----------------------------------------
Training Random Forest...
Accuracy for Random Forest: 0.6924528301886792
----------------------------------------
Training KNN...
Accuracy for KNN: 0.37169811320754714
----------------------------------------
The best model is Random Forest with accuracy 0.6924528301886792


In [30]:
best_pipeline = Pipeline(model=best_model, **models[best_model])

# Fit the best model on the training data
best_pipeline.fit(X_train, y_train)

# Make predictions on test_df using the best model
test_predictions = best_pipeline.predict(test)

# Display the predicted labels
print("Predicted labels for test_df:")
print(test_predictions)

Predicted labels for test_df:
[ 0  0  2  2  8 10  2  0  9  8  1  6  6  0 10  2 10  7  0 10  3  9  2  4
  0  0  0  2  8  3 10  7  3  3  3  8 10 10  3  8  8  0  2  9  0  6  0  3
 10  7  6  0  3 10  6  3  0  0  1  7  8  8  3  0  7  3  2  8  0  7  7  8
  4  6  5  0  8  2  2  4 10  8  9  2  0  4  5  0  7  4  8  0  9  7  2  0
  3  8 10  8  3  8  3  6  7  0  2  2 10  4 10  7 10  7  8  8 10  0  7  8
  8  0  4  6  1 10  7  2  0  7  3  0  8  3  9  3  8  7  7  0  4  2  4  2
  8  0  7  3  2  9  0  7  4  9  0  3  7  0 10  0  8  0 10  0  0  4  4  0
  1  2  9 10  2 10  7  3  0  3  8  7  0  6  0  6  1  4  9  0  0  8  3  7
  2  9  9  9  9  3 10  3  6  0  0  9  8  1  2 10  5  7  4  6  8  3  0  0
  0 10  6  0  9  2  0  2  0  8  8  3  0  3  4  7 10  7  3  0  9  2  0  0
  2  0  7  2  3  1  3  8  7 10  8  0  7  8  0  9  8  8  4  3  3  3  8  7
  7 10  0  3  8  8 10  8  4  9 10  9  7  4  4  0  1  0  6  8  8 10  9  4
  1  0  7 10  8  0  8 10  9 10 10  4  3  2  8]


In [34]:
predictions_df = pd.DataFrame({'id': test['id'], 'prognosis': test_predictions})
print(predictions_df.tail(10))

       id  prognosis
293  1000          0
294  1001          8
295  1002         10
296  1003          9
297  1004         10
298  1005         10
299  1006          4
300  1007          3
301  1008          2
302  1009          8


In [38]:
test['prognosis'] = test_predictions
test[['id', 'prognosis']].to_csv('submission.csv', index=False)

## Conclusion

In this project, we explored various machine learning models to classify vector-borne diseases using a tabular dataset. The models tested included Decision Tree, Random Forest, and K-Nearest Neighbors (KNN). Below are the observed performances:

- **Decision Tree**: Accuracy of 40.19%
- **KNN**: Accuracy of 37.17%
- **Random Forest**: Accuracy of 69.25%

Among these, the **Random Forest** model emerged as the best-performing model with an accuracy of **69.25%** on the validation set. The same model achieved a Kaggle leaderboard score of **62.08%**, demonstrating its robustness and generalization capabilities.

Future improvements could involve hyperparameter tuning, feature engineering, or experimenting with more advanced models to further enhance the predictive accuracy. This project highlights the potential of Random Forest in handling classification tasks with moderate complexity.
